In [ ]:
# SETUP: pin numpy<2.0 for torchvision 0.17.2 compatibility
# torchvision==0.17.2 was compiled against NumPy 1.x; NumPy 2.x breaks the ABI.
# This cell installs numpy<2.0 and restarts the kernel once if needed.
import subprocess, sys

def _numpy_ok():
    try:
        import numpy as np
        major = int(np.__version__.split('.')[0])
        return major < 2
    except Exception:
        return False

if not _numpy_ok():
    print('Installing numpy<2.0 ...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'numpy<2.0', '--upgrade'])
    print('Done. Restarting kernel ...')
    import IPython
    IPython.get_ipython().kernel.do_shutdown(restart=True)
else:
    import numpy as np
    print('numpy', np.__version__, '-- OK (< 2.0)')


# Replicación del paper: PCNN + ELM para detección de Diabetic Retinopathy

Este notebook implementa una réplica del modelo propuesto en:

Nahiduzzaman et al. (2023), *Diabetic retinopathy identification using parallel convolutional neural network based feature extractor and ELM classifier*, Expert Systems with Applications, 217, 119557.

## Arquitectura propuesta en el paper

El modelo consta de dos componentes principales:

1. **Parallel CNN (PCNN)** como extractor de características:
   - 4 ramas paralelas
   - 64 filtros cada una
   - Kernels: 9×9, 7×7, 5×5, 3×3
   - MaxPooling después de cada rama
   - Capas convolucionales adicionales
   - FC layers → 120 features finales

2. **Extreme Learning Machine (ELM)** como clasificador:
   - Pesos de entrada aleatorios
   - Función de activación ReLU
   - Solución cerrada usando pseudoinversa o ridge regression
   - No requiere backpropagation

Este notebook integra el modelo dentro del framework Lightning del proyecto SAM-AI.

# PCNN + ELM como modelo registrado (SAM-AI)

Este notebook crea (si no existe) el archivo `sam_ml/modeling/models/pcnn_elm_lightning.py`, lo registra vía import side-effect y entrena/evalúa con `get_model('pcnn_elm')` usando DDR2019 procesado en `data/processed/ddr2019/`.


## Implementación del PCNN (Parallel CNN)

Siguiendo el paper:

- Se implementan 4 ramas paralelas con diferentes tamaños de kernel:
  - 9×9
  - 7×7
  - 5×5
  - 3×3

El objetivo es capturar patrones a múltiples escalas espaciales en imágenes de retina.

Posteriormente:
- Concatenación de mapas de características
- Convoluciones adicionales
- Dropout (paper usa 0.5)
- Reducción a 120 features finales

In [ ]:
# Four parallel convolution branches (multi-scale feature extraction)
# Inspired directly by the PCNN architecture described in the paper

from __future__ import annotations

import random
from pathlib import Path

import numpy as np
import torch

def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\Esmer\OneDrive\Documentos\GitHub\sam-ai\.venv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\Esmer\OneDrive\Documentos\GitHub\sam-ai\.venv\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\Esmer\OneDrive\Documentos\GitHub\sam-ai\.venv\Lib\site-packages\ipykernel\kernelapp.py", 

device(type='cpu')

## 1) Detectar root del repo y validar DDR2019 procesado

In [2]:
def find_project_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    for parent in [p, *p.parents]:
        if (parent / "pyproject.toml").exists() and (parent / "sam_ml").exists():
            return parent
    return p

PROJECT_ROOT = find_project_root()
PROJECT_ROOT


WindowsPath('C:/Users/Esmer/OneDrive/Documentos/GitHub/sam-ai')

In [3]:
data_dir = PROJECT_ROOT / "data" / "processed" / "ddr2019"
labels_csv = data_dir / "labels.csv"
images_dir = data_dir / "images"

assert data_dir.exists(), f"Missing: {data_dir}"
assert labels_csv.exists(), f"Missing: {labels_csv}"
assert images_dir.exists(), f"Missing: {images_dir}"

print("OK:", data_dir)


OK: C:\Users\Esmer\OneDrive\Documentos\GitHub\sam-ai\data\processed\ddr2019


## 2) Crear `sam_ml/modeling/models/pcnn_elm_lightning.py` (si no existe)

## Extreme Learning Machine (ELM)

ELM es una red neuronal de una sola capa oculta donde:

- Los pesos de entrada son inicializados aleatoriamente.
- No se actualizan mediante backpropagation.
- Los pesos de salida se calculan mediante solución cerrada:

    β = (HᵀH + λI)⁻¹ HᵀT

Donde:
- H = activaciones de la capa oculta
- T = etiquetas one-hot
- λ = parámetro de regularización (ridge)

Esto reduce drásticamente el tiempo de entrenamiento comparado con redes tradicionales.

In [ ]:
# The original paper uses fixed image size.
# Here we use AdaptiveAvgPool2d to make the model resolution-independent.
import textwrap

model_file = PROJECT_ROOT / "sam_ml" / "modeling" / "models" / "pcnn_elm_lightning.py"
model_file.parent.mkdir(parents=True, exist_ok=True)

if not model_file.exists():
    model_py = """from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Optional, Tuple, List

import torch
import torch.nn as nn
import torch.nn.functional as F

from sam_ml.modeling.models.base import BaseLightningModel
from sam_ml.modeling.models.registry import register_model


class _ConvBNReLUPool(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, k: int, padding: int, pool: bool = True) -> None:
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, kernel_size=k, padding=padding, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2) if pool else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv(x)
        x = self.bn(x)
        x = self.act(x)
        x = self.pool(x)
        return x


class PCNNFeatureExtractor(nn.Module):
    '''
    Lightweight Parallel CNN inspired by the paper:
    - 4 parallel conv blocks with 64 filters and kernels 9,7,5,3 (same padding)
    - concat -> conv(32,k=3,valid) -> pool -> conv(16,k=3,valid) -> pool -> dropout
    - flatten -> dense(250) -> dropout -> dense(120)  (features)

    Uses AdaptiveAvgPool2d to avoid dependence on input image size (your pipeline may be 512x512).
    '''

    def __init__(self, feature_dim: int = 120, dropout_p: float = 0.5) -> None:
        super().__init__()
        self.p9 = _ConvBNReLUPool(3, 64, k=9, padding=9 // 2, pool=True)
        self.p7 = _ConvBNReLUPool(3, 64, k=7, padding=7 // 2, pool=True)
        self.p5 = _ConvBNReLUPool(3, 64, k=5, padding=5 // 2, pool=True)
        self.p3 = _ConvBNReLUPool(3, 64, k=3, padding=3 // 2, pool=True)

        self.conv5 = nn.Conv2d(256, 32, kernel_size=3, padding=0, bias=False)
        self.bn5 = nn.BatchNorm2d(32)
        self.conv6 = nn.Conv2d(32, 16, kernel_size=3, padding=0, bias=False)
        self.bn6 = nn.BatchNorm2d(16)

        self.act = nn.ReLU(inplace=True)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.dp_after_conv = nn.Dropout(p=dropout_p)

        # Stabilize FC input size (independent of input resolution)
        self.adapt = nn.AdaptiveAvgPool2d((29, 29))

        self.fc1 = nn.Linear(16 * 29 * 29, 250)
        self.bn_fc1 = nn.BatchNorm1d(250)
        self.dp_after_fc1 = nn.Dropout(p=dropout_p)
        self.fc_features = nn.Linear(250, feature_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        a = self.p9(x)
        b = self.p7(x)
        c = self.p5(x)
        d = self.p3(x)

        x = torch.cat([a, b, c, d], dim=1)

        x = self.conv5(x)
        x = self.bn5(x)
        x = self.act(x)
        x = self.pool(x)

        x = self.conv6(x)
        x = self.bn6(x)
        x = self.act(x)
        x = self.pool(x)

        x = self.dp_after_conv(x)
        x = self.adapt(x)

        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = self.bn_fc1(x)
        x = self.act(x)
        x = self.dp_after_fc1(x)

        feats = self.fc_features(x)
        return feats


@dataclass
class ELMState:
    w: torch.Tensor
    b: torch.Tensor
    beta: torch.Tensor
    mean: torch.Tensor
    std: torch.Tensor

# Standardize features (mean=0, std=1)
# Improves numerical stability of matrix inversion

def _fit_elm_closed_form(
    x: torch.Tensor,
    y: torch.Tensor,
    hidden_dim: int,
    num_classes: int,
    ridge_lambda: float,
    device: torch.device,
    seed: int = 42,
) -> ELMState:
    g = torch.Generator(device="cpu")
    g.manual_seed(seed)

    x = x.to(device)
    y = y.to(device)

    mean = x.mean(dim=0)
    std = x.std(dim=0).clamp_min(1e-6)
    x_std = (x - mean) / std

    w = torch.randn(x_std.shape[1], hidden_dim, generator=g, device=device) * 0.1
    b = torch.randn(hidden_dim, generator=g, device=device) * 0.1

    h = F.relu(x_std @ w + b)
    t = F.one_hot(y, num_classes=num_classes).float()

    ht_h = h.T @ h
    ht_t = h.T @ t
    eye = torch.eye(hidden_dim, device=device, dtype=ht_h.dtype)
    beta = torch.linalg.solve(ht_h + ridge_lambda * eye, ht_t)

    return ELMState(w=w, b=b, beta=beta, mean=mean, std=std)


def _elm_predict_logits(state: ELMState, x: torch.Tensor) -> torch.Tensor:
    x_std = (x - state.mean) / state.std
    h = F.relu(x_std @ state.w + state.b)
    return h @ state.beta


class PCNNELMLightning(BaseLightningModel):
    '''
    Train CNN features with a linear head (CrossEntropy).
    Fit ELM at the end of each epoch using detached train features.
    Validation/Test uses ELM logits if available, otherwise uses linear head.
    '''

    def __init__(
        self,
        num_classes: int = 5,
        learning_rate: float = 1e-4,
        optimizer: str = "adam",
        weight_decay: float = 0.0,
        elm_hidden_dim: int = 1000,
        elm_ridge_lambda: float = 1e-3,
        elm_seed: int = 42,
        **kwargs: Any,
    ) -> None:
        self.elm_hidden_dim = elm_hidden_dim
        self.elm_ridge_lambda = elm_ridge_lambda
        self.elm_seed = elm_seed

        self._elm_state: Optional[ELMState] = None
        self._train_feats: List[torch.Tensor] = []
        self._train_labels: List[torch.Tensor] = []

        super().__init__(
            num_classes=num_classes,
            learning_rate=learning_rate,
            optimizer=optimizer,
            weight_decay=weight_decay,
            **kwargs,
        )

    def _create_model(self) -> None:
        self.backbone = PCNNFeatureExtractor(feature_dim=120, dropout_p=0.5)
        self.linear_head = nn.Linear(120, self.num_classes)
        self.model = nn.Sequential(self.backbone, self.linear_head)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feats = self.backbone(x)
        if self._elm_state is not None:
            return _elm_predict_logits(self._elm_state, feats)
        return self.linear_head(feats)

    def training_step(self, batch: Tuple[torch.Tensor, torch.Tensor], batch_idx: int) -> torch.Tensor:
        x, y = batch
        feats = self.backbone(x)
        logits = self.linear_head(feats)
        loss = self.criterion(logits, y)

        metrics = self._compute_metrics(logits.detach(), y)
        self.log("train_loss", loss, on_step=True, on_epoch=True, prog_bar=True)
        self.log("train_accuracy", metrics["accuracy"], on_step=True, on_epoch=True, prog_bar=True)

        self._train_feats.append(feats.detach().cpu())
        self._train_labels.append(y.detach().cpu())
        return loss

    def on_train_epoch_end(self) -> None:
        if not self._train_feats:
            return

        x = torch.cat(self._train_feats, dim=0)
        y = torch.cat(self._train_labels, dim=0)

        self._elm_state = _fit_elm_closed_form(
            x=x,
            y=y,
            hidden_dim=self.elm_hidden_dim,
            num_classes=self.num_classes,
            ridge_lambda=self.elm_ridge_lambda,
            device=self.device,
            seed=self.elm_seed,
        )

        self._train_feats.clear()
        self._train_labels.clear()

    def validation_step(self, batch: Tuple[torch.Tensor, torch.Tensor], batch_idx: int) -> None:
        x, y = batch
        logits = self.forward(x)
        loss = self.criterion(logits, y)

        metrics = self._compute_metrics(logits.detach(), y)
        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log("val_accuracy", metrics["accuracy"], on_step=False, on_epoch=True, prog_bar=True)

    def test_step(self, batch: Tuple[torch.Tensor, torch.Tensor], batch_idx: int) -> None:
        x, y = batch
        logits = self.forward(x)
        loss = self.criterion(logits, y)

        metrics = self._compute_metrics(logits.detach(), y)
        self.log("test_loss", loss, on_step=False, on_epoch=True)
        self.log("test_accuracy", metrics["accuracy"], on_step=False, on_epoch=True)


@register_model("pcnn_elm")
def create_pcnn_elm_model(**kwargs: Any) -> BaseLightningModel:
    return PCNNELMLightning(**kwargs)
"""
    model_file.write_text(textwrap.dedent(model_py), encoding="utf-8")
    print("Created:", model_file)
else:
    print("Already exists:", model_file)


Already exists: C:\Users\Esmer\OneDrive\Documentos\GitHub\sam-ai\sam_ml\modeling\models\pcnn_elm_lightning.py


## 3) Registrar modelo (import side-effect) y verificar `list_models()`

## Hybrid Training Strategy

Para mantener compatibilidad con PyTorch Lightning:

1. Se entrena el backbone CNN usando una cabeza lineal (CrossEntropy).
2. Al final de cada epoch:
   - Se extraen features del conjunto de entrenamiento.
   - Se entrena el clasificador ELM usando solución cerrada.
3. Durante validación y test:
   - Si el ELM ya fue entrenado, se usa para inferencia.
   - Si no, se usa la cabeza lineal.

In [5]:
from sam_ml.modeling.models import get_model, list_models
from sam_ml.modeling.models import pcnn_elm_lightning  # noqa: F401

models = list_models()
print(models)
assert "pcnn_elm" in models, "pcnn_elm no se registró (revisa el archivo/imports)"


['simple_cnn', 'pcnn_elm']


## 4) Dataset + DataLoaders (DDR2019Dataset)

In [ ]:
from torch.utils.data import DataLoader
from sam_ml.datasets import DDR2019Dataset
from sam_ml.config import get_model_config, get_training_config
import torchvision.transforms.v2 as T2

# FIX: Use transforms.v2 to avoid torch.from_numpy() crash with NumPy 2.x
safe_transform = T2.Compose([
    T2.ToImage(),                           # PIL -> tv_tensors.Image (no numpy)
    T2.ToDtype(torch.float32, scale=True),  # uint8 -> float32 [0, 1]
])

model_cfg = get_model_config()
train_cfg = get_training_config()

TRAIN_RATIO = 0.8
VAL_RATIO = 0.2
RANDOM_STATE = 42

train_ds = DDR2019Dataset(data_dir, split="train", train_ratio=TRAIN_RATIO, val_ratio=VAL_RATIO, random_state=RANDOM_STATE, transform=safe_transform)
val_ds   = DDR2019Dataset(data_dir, split="val",   train_ratio=TRAIN_RATIO, val_ratio=VAL_RATIO, random_state=RANDOM_STATE, transform=safe_transform)

train_loader = DataLoader(
    train_ds,
    batch_size=train_cfg.batch_size,
    shuffle=True,
    num_workers=0,
)

val_loader = DataLoader(
    val_ds,
    batch_size=train_cfg.batch_size,
    shuffle=False,
    num_workers=0,
)
print("train:", len(train_ds), "val:", len(val_ds), "num_classes:", model_cfg.num_classes)


## 5) Entrenar `pcnn_elm` con PyTorch Lightning

## Entrenamiento

Aunque el paper no utiliza backpropagation en la etapa ELM,
en esta implementación el backbone CNN se entrena de forma supervisada
para mejorar la calidad de las representaciones.

Esto permite comparar de forma justa contra modelos end-to-end.

In [7]:
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

model = get_model(
    "pcnn_elm",
    num_classes=model_cfg.num_classes,
    learning_rate=model_cfg.learning_rate,
    optimizer=model_cfg.optimizer,
    weight_decay=model_cfg.weight_decay,
    elm_hidden_dim=1000,
    elm_ridge_lambda=1e-3,
    elm_seed=42,
)

ckpt_cb = ModelCheckpoint(
    monitor="val_accuracy",
    mode="max",
    save_top_k=1,
    filename="pcnn_elm-{epoch:02d}-{val_accuracy:.4f}",
)
es_cb = EarlyStopping(monitor="val_accuracy", mode="max", patience=5)

trainer = pl.Trainer(
    max_epochs=train_cfg.num_epochs,
    accelerator="auto",
    devices=1,
    log_every_n_steps=10,
    callbacks=[ckpt_cb, es_cb],
)

trainer.fit(model, train_loader, val_loader)
print("Best checkpoint:", ckpt_cb.best_model_path)


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
C:\Users\Esmer\OneDrive\Documentos\GitHub\sam-ai\.venv\Lib\site-packages\pytorch_lightning\trainer\connectors\logger_connector\logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default

  | Name        | Type                 | Params | Mode  | FLOPs
---------------------------------------------------------------------
0 | criterion   | CrossEntropyLoss     | 0      | train | 0    
1 | backbone    | PCNNFeatureExtractor | 3.5 M  | train | 0    
2 | linear_head | Linear               | 605    | train | 0    
3 | model       | Sequential           | 

Sanity Checking: |                                                                                            …

C:\Users\Esmer\OneDrive\Documentos\GitHub\sam-ai\.venv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


RuntimeError: Numpy is not available

## 6) Evaluación (val)

## Interpretación de Resultados

El paper reporta mejoras en precisión y velocidad de entrenamiento
al usar ELM sobre un clasificador totalmente conectado tradicional.

Aspectos a analizar en esta réplica:

- Accuracy global
- Accuracy por clase
- Comparación contra simple_cnn
- Tiempo por epoch
- Estabilidad del entrenamiento

Diferencias pueden surgir por:
- Dataset preprocessing
- Tamaño de imagen
- Regularización λ
- Número de neuronas ocultas en ELM

In [ ]:
val_results = trainer.validate(model, val_loader, verbose=False)
val_results


## 7) Comparación rápida vs `simple_cnn` (opcional)

In [ ]:
# Import side-effect to register simple_cnn
from sam_ml.modeling.models import simple_cnn_lightning  # noqa: F401

simple = get_model(
    "simple_cnn",
    num_classes=model_cfg.num_classes,
    learning_rate=model_cfg.learning_rate,
    optimizer=model_cfg.optimizer,
    weight_decay=model_cfg.weight_decay,
)

trainer2 = pl.Trainer(
    max_epochs=train_cfg.num_epochs,
    accelerator="auto",
    devices=1,
    log_every_n_steps=10,
)

trainer2.fit(simple, train_loader, val_loader)
trainer2.validate(simple, val_loader, verbose=False)
